# Amparo -- M3: busqueda v2 y dataset v2 con el contexto real

Tres cosas en una sesion de GPU (T4 alcanza, unos 15 minutos):

1. **Reconstruir el indice** con el corpus de 36 normas (unos 12 000 chunks) y
   los arreglos del 2026-10-08: la Constitucion ya no pierde los articulos que
   abrian pagina (el 23, derecho de peticion, estaba pegado al 22), el Codigo
   Sustantivo del Trabajo trae la numeracion oficial, y cada chunk se embebe con
   su capitulo.
2. **Medir la busqueda** sin juez: los 45 gold del eval set tienen etiquetados
   los articulos que los responden (`data/eval_set_articulos.json`), y
   `tools/rag/benchmark_busqueda.py` cuenta en cuantos el buscador trae uno en su
   top-k. Compara denso con y sin el **enrutador por categoria**
   (`tools/rag/enrutador.py`) y barre el piso de la valvula de escape.
3. **Regenerar `data/dataset_v2.jsonl`** con los fragmentos que de verdad trae
   e5 + FAISS + enrutador (en local se arma con BM25).

Todo queda en Drive, en `rag/busqueda_v2/`. Lo que hay que traer de vuelta al
repo: `results/busqueda_<fecha>.json`, `data/dataset_v2.jsonl`,
`data/dataset_m1_v2.jsonl` y `run_manifest.json`.

El indice nuevo se copia a `rag/busqueda_v2/` y **no pisa** el de `rag/`, que
es el que usaron las corridas del 7 de octubre.

**Secretos:** no hace falta ninguno (no usa juez ni W&B).


In [1]:
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo


Cloning into 'Amparo'...
remote: Enumerating objects: 1410, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 1410 (delta 106), reused 202 (delta 87), pack-reused 1087 (from 1)
Receiving objects: 100% (1410/1410), 10.15 MiB | 26.45 MiB/s, done.
Resolving deltas: 100% (867/867), done.
/content/Amparo


In [2]:
import os, subprocess, sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/enrutador.py",
    "tools/rag/benchmark_busqueda.py",
    "tools/dataset_v2.py",
    "data/eval_set_articulos.json",
    "data/corpus/normas/servicios_publicos_domiciliarios_ley_142_1994.md",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(f"Falta en la rama '{RAMA}': {faltan}. Hay que hacer push de los commits del 2026-10-08.")
print("Repo OK:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)


Repo OK: c336fc2 M3/M1: dejar listos para correr los notebooks de busqueda v2 y de fine-tuning



In [3]:
!pip install -q -r requirements.txt
!pip install -q -U transformers sentence-transformers


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 48.8 MB/s eta 0:00:00


## Fase 1 -- Reconstruir el indice

`ingest -> chunk -> embed -> store` con e5. El indice nuevo reemplaza al de S07
en `artifacts/` y se copia a Drive en `rag/busqueda_v2/` (no pisa el de
`rag/`, que es el que usaron las corridas del 7 de octubre).


In [4]:
from tools.rag import config, pipeline

store = pipeline.build_index()
print(f"Chunks indexados: {len(store)}")

from google.colab import drive
import pathlib, shutil

drive.mount("/content/drive")
DESTINO = pathlib.Path(config.DRIVE_ROOT) / "rag" / "busqueda_v2"
DESTINO.mkdir(parents=True, exist_ok=True)
for archivo in (config.FAISS_INDEX_PATH, config.FAISS_METADATA_PATH):
    shutil.copy(archivo, DESTINO / archivo.name)
print("Indice copiado a", DESTINO)


[1/4] ingest...
[2/4] chunk...
      36 documentos -> 11975 chunks
[3/4] embed (intfloat/multilingual-e5-base)...


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      250/11975 (2%) -- ETA ~22.7 min
      500/11975 (4%) -- ETA ~11.5 min
      750/11975 (6%) -- ETA ~7.8 min
      1000/11975 (8%) -- ETA ~5.9 min
      1250/11975 (10%) -- ETA ~4.7 min
      1500/11975 (13%) -- ETA ~4.0 min
      1750/11975 (15%) -- ETA ~3.4 min
      2000/11975 (17%) -- ETA ~3.0 min
      2250/11975 (19%) -- ETA ~2.7 min
      2500/11975 (21%) -- ETA ~2.4 min
      2750/11975 (23%) -- ETA ~2.2 min
      3000/11975 (25%) -- ETA ~2.0 min
      3250/11975 (27%) -- ETA ~1.9 min
      3500/11975 (29%) -- ETA ~1.7 min
      3750/11975 (31%) -- ETA ~1.6 min
      4000/11975 (33%) -- ETA ~1.5 min
      4250/11975 (35%) -- ETA ~1.4 min
      4500/11975 (38%) -- ETA ~1.3 min
      4750/11975 (40%) -- ETA ~1.2 min
      5000/11975 (42%) -- ETA ~1.1 min
      5250/11975 (44%) -- ETA ~1.0 min
      5500/11975 (46%) -- ETA ~1.0 min
      5750/11975 (48%) -- ETA ~0.9 min
      6000/11975 (50%) -- ETA ~0.9 min
      6250/11975 (52%) -- ETA ~0.8 min
      6500/11975 (54%) -- ETA 

In [5]:
# Manifiesto: este indice NO es el de las corridas del 7 de octubre (36 normas en
# vez de 10, ~12 000 chunks en vez de 3 420, Constitucion y CST corregidos). Sin
# el hash no hay como saber despues con cual se midio cada cifra, y esa confusion
# ya costo un dia: el adaptador de Drive se sobrescribio el 6 de octubre y las
# corridas anteriores quedaron sin forma de reproducirse.
import pathlib, subprocess

from tools.rag import manifiesto

try:
    _hw = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    _hw = "(sin GPU)"

_m = manifiesto.construir(
    "busqueda_v2",
    use_lora=False,                       # esta corrida no genera, solo recupera
    n_chunks=len(store.metadata),
    n_eval_set=len(registros) if "registros" in globals() else None,
    hardware=_hw,
)
print("Manifiesto:", _m.guardar(DESTINO))
print(f"  indice {_m.hash_indice} | corpus {_m.hash_corpus} | eval set {_m.hash_eval_set}")
print(f"  chunks {_m.n_chunks}")

Manifiesto: /content/drive/MyDrive/Colab Notebooks/Amparo/rag/busqueda_v2/run_manifest.json
  indice 19657d22583f93d0 | corpus 1c552822959e2932 | eval set a5151999c2095d00
  chunks 11975


## Fase 2 -- ¿La busqueda trae el articulo?

Acierto@k: en cuantos de los 45 gold el buscador trae, entre sus k primeros, al
menos un articulo de los etiquetados. MRR: el promedio de 1/puesto del primer
acierto.

Referencia local (BM25, sin modelos): acierto@5 de 0.22 sin enrutador y 0.33
con enrutador. La pregunta de esta fase es si el enrutador tambien mejora e5,
y cual configuracion queda en `config.py`.

El barrido de pisos dice, para cada umbral de coseno, cuantos gold se quedan
sin contexto (el sistema se niega a responder algo que si podia) y cuantos
adversariales tambien. El piso actual (0.82) dejaba 8 de 75 consultas sin
contexto el 7 de octubre, 5 de ellas gold.


In [6]:
!python -m tools.rag.benchmark_busqueda --indice
!cp results/busqueda_*.json "{DESTINO}/"


Loading weights: 100% 199/199 [00:00<00:00, 5851.40it/s]
configuracion                @1     @3     @5    @10    mrr
A denso                    0.22   0.29   0.33   0.51   0.28
A denso+enrutador          0.18   0.36   0.47   0.56   0.28
B hybrid+enrutador         0.29   0.42   0.49   0.60   0.38
config.json: 100% 891/891 [00:00<00:00, 5.12MB/s]

model.safetensors: downloading bytes:  87% 410M/471M [00:01<00:00, 362MB/s, 36.1MB/s  ]
model.safetensors: downloading bytes: 100% 426M/426M [00:02<00:00, 183MB/s, 38.4MB/s  ]
model.safetensors: reconstructing file: 100% 471M/471M [00:02<00:00, 202MB/s, 44.5MB/s  ]
Loading weights: 100% 201/201 [00:00<00:00, 4201.49it/s]
tokenizer_config.json: 100% 435/435 [00:00<00:00, 2.36MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.28MB/s]
C rerank+enrutador         0.24   0.47   0.53   0.64   0.36

pisos_denso:
   {'piso': None, 'gold_sin_contexto': '0/45', 'adversariales_sin_contexto': '0/30', 'gold_acierto@5': '15/45'}
   {'piso': 0.78, 'go

> **Que decidir con la tabla.**
>
> 1. **Enrutador.** Si `A denso+enrutador` supera a `A denso` en acierto@5, el
>    enrutador sirve. Hoy `USE_ENRUTADOR = False` en `tools/rag/config.py`, igual
>    que `USE_HYBRID` y `USE_RERANK`: la configuracion A tiene que seguir siendo
>    denso puro para poder comparar esta corrida contra la del 2026-10-07 y
>    separar cuanto aporto el corpus de cuanto el enrutador.
> 2. **El piso.** El mas alto que no deje gold sin contexto teniendo acierto. El
>    actual (0.82) dejaba 8 de 75 consultas sin contexto el 7 de octubre, 5 de
>    ellas gold.
>
> Las dos decisiones se aplican en la celda siguiente, y despues se anotan en
> `tools/rag/config.py` y en `docs/m3_decisiones_rag.md`.

In [7]:
# La fase 3 arma el contexto del dataset con la busqueda de PRODUCCION, asi que
# la decision de la fase 2 tiene que estar aplicada antes de correrla: si no, se
# entrena con un contexto que no es el que el sistema va a ver.
#
# Editar segun la tabla de arriba y volver a correr esta celda.
from tools.rag import config

config.USE_ENRUTADOR = False        # True si el enrutador gano en acierto@5
config.RETRIEVAL_MIN_SCORE = 0.82   # el piso elegido en el barrido

print(f"enrutador: {config.USE_ENRUTADOR} | piso: {config.RETRIEVAL_MIN_SCORE}")
print("Si cambiaste algo, deja el mismo valor en tools/rag/config.py del repo:")
print("  de lo contrario la proxima corrida vuelve al valor viejo.")

enrutador: False | piso: 0.82
Si cambiaste algo, deja el mismo valor en tools/rag/config.py del repo:
  de lo contrario la proxima corrida vuelve al valor viejo.


## Fase 3 -- Dataset v2 con el contexto de e5

Mismas fuentes (`data/dataset_src_v2/`) y mismas puertas de calidad; cambia de
donde salen los fragmentos que acompanan al articulo que responde: los trae la
busqueda de produccion. Si las puertas no pasan, no escribe nada.


In [8]:
!python -m tools.dataset_v2 --indice
!cp data/dataset_v2.jsonl "{DESTINO}/"
import json, collections
regs = [json.loads(l) for l in open("data/dataset_v2.jsonl", encoding="utf-8")]
print(len(regs), "ejemplos |", collections.Counter(r["contexto_origen"] for r in regs))


Loading weights: 100% 199/199 [00:00<00:00, 5102.00it/s]
Dataset v2: 755 ejemplos | por modo: {'B1': 391, 'B2': 263, 'B3': 101}
Categorias: 27

PUERTAS DE CALIDAD
  [OK  ] contexto vacio: 0
  [OK  ] longitud: 0
  [OK  ] promesas de resultado: 0
  [OK  ] urgencia sin ayuda inmediata: 0
  [OK  ] ruta incorrecta: 0
  [OK  ] entidad inventada: 0
  [OK  ] mecanismo legal: 0
  [OK  ] escape: 0
  [OK  ] cita en escape: 0
  [OK  ] cita no respaldada: 0
  [OK  ] no cita la fuente: 0
  [OK  ] plazo sin respaldo: 0
  [OK  ] B3 sin la parte que falta: 0
  [OK  ] escapes (B2) >= 1 de cada 3: 263/755
  [OK  ] repeticion: 0
  [OK  ] fuga al eval set: 0
  [OK  ] articulo fuente de mas de 3 ejemplos: 0
  [OK  ] preguntas casi iguales: 0
  [OK  ] ids duplicados: 0

Escrito /content/Amparo/data/dataset_v2.jsonl (755 ejemplos)
755 ejemplos | Counter({'busqueda+oraculo': 414, 'busqueda-otras-normas': 263, 'busqueda': 78})


In [9]:
# El combinado que entrena M1 lleva dentro los contextos de dataset_v2, asi que
# hay que rehacerlo despues de regenerarlo con e5. Si no, se entrena con los
# contextos de BM25 y el notebook de M1 no tendria como notarlo.
!python -m tools.dataset_m1_v2
!cp data/dataset_m1_v2.jsonl "{DESTINO}/"

  v1  train  1305   val  231
  v2  train   652   val  103
  total 2291
  modos de v2: {'B1': 391, 'B3': 101, 'B2': 263}

  todas las comprobaciones pasan

Escrito /content/Amparo/data/dataset_m1_v2.jsonl (2291 ejemplos)
